# Parking Occupancy Survey Analysis — TU Viertel (TDM Seminar)

This notebook analyzes on-street parking occupancy survey data collected as part of the **TDM Seminar: TU Viertel** project (Chair of Urban Structure and Transport Planning, TUM).

Two survey rounds were conducted (Round 1: April 2023, Round 2: June 2023), each covering a weekday and a weekend day, across 9 street sections in the TU Viertel neighborhood in Munich. For each section, on-street parking occupancy (%) was recorded at multiple time slots (morning, midday/afternoon, evening).

This notebook:
1. Loads and consolidates the raw survey data (originally recorded across 4 Excel workbooks)
2. Computes average occupancy rates per street section
3. Compares occupancy between Round 1 vs Round 2, and Weekday vs Weekend
4. Reproduces the key comparison charts presented in the final seminar presentation


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('parking_occupancy_data.csv')
df.head()

## Street Sections Surveyed

In [ ]:
street_lookup = df[['street_id','street_name']].drop_duplicates().sort_values('street_id')
street_lookup

## Average Occupancy Rate per Section: Round 1 vs Round 2

In [ ]:
avg = df.groupby(['round','street_id','street_name'])['occupancy_pct'].mean().reset_index()
pivot = avg.pivot(index=['street_id','street_name'], columns='round', values='occupancy_pct').reset_index()
pivot

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
x = pivot['street_id']
ax.bar(x - 0.2, pivot['round1'], width=0.4, label='Round 1 (April)')
ax.bar(x + 0.2, pivot['round2'], width=0.4, label='Round 2 (June)')
ax.set_xticks(x)
ax.set_xlabel('Street Section')
ax.set_ylabel('Average Occupancy Rate (%)')
ax.set_title('Average Parking Occupancy Rate per Section: Round 1 vs Round 2')
ax.legend()
ax.set_ylim(0,100)
for i, (r1, r2) in enumerate(zip(pivot['round1'], pivot['round2'])):
    ax.text(x[i]-0.2, r1+1, f"{r1:.0f}%", ha='center', fontsize=8)
    ax.text(x[i]+0.2, r2+1, f"{r2:.0f}%", ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('occupancy_comparison_rounds.png', dpi=150)
plt.show()

**Observation:** Most sections show a *decrease* in average occupancy from Round 1 (April) to Round 2 (June).
Possible explanations discussed in the project: warmer weather encouraging a modal shift away from private cars toward public transport, walking, and cycling.

## Average Occupancy Rate per Section: Weekday vs Weekend

In [ ]:
avg2 = df.groupby(['day_type','street_id','street_name'])['occupancy_pct'].mean().reset_index()
pivot2 = avg2.pivot(index=['street_id','street_name'], columns='day_type', values='occupancy_pct').reset_index()
pivot2

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
x = pivot2['street_id']
ax.bar(x - 0.2, pivot2['weekday'], width=0.4, label='Weekday')
ax.bar(x + 0.2, pivot2['weekend'], width=0.4, label='Weekend')
ax.set_xticks(x)
ax.set_xlabel('Street Section')
ax.set_ylabel('Average Occupancy Rate (%)')
ax.set_title('Average Parking Occupancy Rate per Section: Weekday vs Weekend (Both Rounds)')
ax.legend()
ax.set_ylim(0,100)
plt.tight_layout()
plt.savefig('occupancy_comparison_daytype.png', dpi=150)
plt.show()

## Key Findings

- Average occupancy across all sections and time slots ranges from **~18% to ~65%**
- **Section 7 (Zieblandstraße)** consistently shows the highest occupancy (50-65%), indicating high parking demand/pressure
- **Section 8 (Luisenstraße)** consistently shows the lowest occupancy (~19%), suggesting underused parking capacity
- Most sections show **reduced occupancy in Round 2 (June)** compared to Round 1 (April), consistent with a seasonal modal shift toward active mobility (walking/cycling) during warmer weather
- Sections with consistently low occupancy (e.g. Section 8, Section 5/6) are good candidates for **road-space reallocation** — converting underused parking into bike lanes, green space, or community gardens, as recommended in the final project report
